> **SOLUTIONS VERSION.** Try `word2vec_practice.ipynb` first!

# word2vec from scratch — a CS224n A2 clone, made easy

**What you'll build:** the skip-gram word2vec model with (1) naive softmax and (2) negative sampling, the gradients by hand, gradient checking, SGD, training on a tiny corpus, and then the *same thing* with PyTorch and gensim — the way you'd actually do it at work.

**How to use this notebook**
1. Run the cells top to bottom.
2. Each exercise has a `### YOUR CODE HERE` block. Fill it in, then run the test cell right below it.
3. Stuck? Each exercise has a hint. Full solutions are in the companion notebook `word2vec_solutions.ipynb`.

Everything is self-contained: no dataset download, no `utils/` folder. It runs on a laptop or in Google Colab in a couple of minutes.

---
### The whole idea in 4 lines
* Every word gets **two vectors**: $v_w$ (when it's the *center* word) and $u_w$ (when it's an *outside*/context word).
* Score how likely word $o$ appears near word $c$ with a dot product: $u_o^\top v_c$. Big dot product = "these words hang out together".
* Turn scores into probabilities (softmax or sigmoid), compare against what actually happened in the text, and nudge the vectors to reduce the error.
* After training, words used in similar contexts end up with similar vectors. That's the whole trick: *"a word is known by the company it keeps."*

## 0. Setup — helpers (given, nothing to do)
These replace the assignment's `utils/` folder.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

def softmax(x):
    """Numerically stable softmax (works on a 1-D vector or row-wise on a 2-D array).
    Subtracting the max doesn't change the answer but stops exp() from overflowing."""
    x = x - np.max(x, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def normalizeRows(x):
    """Make every row have length 1 (L2 norm)."""
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def gradcheck_naive(f, x, name="", h=1e-4, tol=1e-5):
    """Compare your analytic gradient against a numerical one:
         grad ≈ (f(x + h) - f(x - h)) / 2h     for every single entry of x.
    f must return (loss, grad). The random state is reset before each call so
    negative sampling draws the SAME negatives every time."""
    rndstate = random.getstate()
    random.setstate(rndstate)
    fx, grad = f(x)
    it = np.nditer(x, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        ix = it.multi_index
        old = x[ix]
        x[ix] = old + h; random.setstate(rndstate); fplus, _ = f(x)
        x[ix] = old - h; random.setstate(rndstate); fminus, _ = f(x)
        x[ix] = old
        numgrad = (fplus - fminus) / (2 * h)
        reldiff = abs(numgrad - grad[ix]) / max(1, abs(numgrad), abs(grad[ix]))
        if reldiff > tol:
            print(f"❌ Gradient check FAILED for {name} at index {ix}: "
                  f"yours={grad[ix]:.6f}  numerical={numgrad:.6f}")
            return False
        it.iternext()
    print(f"✅ Gradient check passed for {name}")
    return True

def try_test(test_fn):
    """Runs a test but prints a friendly message if you haven't written the code yet."""
    try:
        test_fn()
    except NotImplementedError:
        print("⏳ Not implemented yet — fill in the YOUR CODE HERE block above, then re-run.")

### A tiny toy corpus
The real assignment uses the Stanford Sentiment Treebank and trains for 1–2 hours. We generate a small corpus with obvious structure (positive words, negative words, royals, people, drinks…) so training takes about a minute and you can actually *see* whether it worked.

In [ ]:
def make_toy_corpus(n_sentences=3000, seed=0):
    rng = random.Random(seed)
    c = rng.choice
    pos    = ["great", "amazing", "wonderful", "excellent", "brilliant"]
    neg    = ["boring", "awful", "terrible", "dull", "bad"]
    things = ["movie", "film", "story", "show"]
    royals = ["king", "queen", "prince", "princess"]
    people = ["man", "woman", "boy", "girl"]
    drinks = ["tea", "coffee", "juice", "milk"]
    templates = [
        lambda: f"the {c(things)} was {c(pos)} and {c(pos)}",
        lambda: f"the {c(things)} was {c(neg)} and {c(neg)}",
        lambda: f"i loved this {c(pos)} {c(things)}",
        lambda: f"i hated this {c(neg)} {c(things)}",
        lambda: f"the {c(royals)} ruled the kingdom from the castle",
        lambda: f"the {c(royals)} wore a golden crown",
        lambda: f"the {c(people)} walked to the shop",
        lambda: f"a {c(people)} bought bread at the shop",
        lambda: f"every morning i drink hot {c(drinks)}",
        lambda: f"a cup of {c(drinks)} please",
    ]
    return [c(templates)() for _ in range(n_sentences)]

class ToyDataset:
    """Same interface the assignment's StanfordSentiment class exposes."""
    def __init__(self, sentences, table_size=100_000):
        self.sentences = [s.split() for s in sentences]
        self.words = sorted({w for s in self.sentences for w in s})
        self.tokens = {w: i for i, w in enumerate(self.words)}     # word -> index
        counts = np.zeros(len(self.words))
        for s in self.sentences:
            for w in s:
                counts[self.tokens[w]] += 1
        # word2vec samples negatives from the unigram distribution raised to 3/4
        probs = counts ** 0.75
        probs /= probs.sum()
        self.table = np.repeat(np.arange(len(self.words)),
                               np.round(probs * table_size).astype(int)).tolist()

    def sampleTokenIdx(self):
        """Draw one word index for negative sampling."""
        return self.table[random.randint(0, len(self.table) - 1)]

    def getRandomContext(self, C):
        """Pick a random center word and the (up to) C words on each side."""
        sent = random.choice(self.sentences)
        i = random.randint(0, len(sent) - 1)
        context = sent[max(0, i - C):i] + sent[i + 1:i + C + 1]
        if not context:
            return self.getRandomContext(C)
        return sent[i], context

dataset = ToyDataset(make_toy_corpus())
print("vocab size:", len(dataset.words))
print("example sentence:", " ".join(dataset.sentences[0]))
random.seed(1); print("example (center, context):", dataset.getRandomContext(2))

## 1. The one notation trap: PDF shapes vs code shapes

| Thing | PDF | This code |
|---|---|---|
| outside vectors | $U$ is $d \times |V|$, **columns** are $u_w$ | `outsideVectors` is `(|V|, d)`, **rows** are $u_w$ (so it's $U^\top$) |
| center vectors | $V$, columns $v_w$ | `centerWordVectors`, rows |
| $v_c$ | column vector $(d)$ | `centerWordVec`, shape `(d,)` |

So every formula from the PDF gets "transposed" when you code it:

| PDF formula | numpy |
|---|---|
| scores $U^\top v_c$ | `outsideVectors @ centerWordVec` → shape `(|V|,)` |
| $\partial J/\partial v_c = U(\hat y - y)$ | `(y_hat - y) @ outsideVectors` → shape `(d,)` |
| $\partial J/\partial U = v_c(\hat y - y)^\top$ | `np.outer(y_hat - y, centerWordVec)` → shape `(|V|, d)` |

**Debugging rule of thumb:** a gradient always has the same shape as the thing it's the gradient of. Print shapes constantly.

## 2. Exercise 1 — sigmoid
$$\sigma(x) = \frac{1}{1+e^{-x}}, \qquad \sigma'(x) = \sigma(x)\,\big(1-\sigma(x)\big)$$
Squashes any number into $(0,1)$, so it can be read as "probability that this (center, outside) pair is real".
Also handy: $1 - \sigma(x) = \sigma(-x)$.

*Hint:* one line, use `np.exp` so it works on arrays too.

In [ ]:
def sigmoid(x):
    ### YOUR CODE HERE (~1 line)
    return 1 / (1 + np.exp(-x))
    ### END YOUR CODE

In [ ]:
def test_sigmoid():
    assert sigmoid(0) == 0.5
    assert np.allclose(sigmoid(np.array([1, 2, 3])), [0.73105858, 0.88079708, 0.95257413])
    print("✅ sigmoid tests passed")
try_test(test_sigmoid)

## 3. Exercise 2 — naive softmax loss and gradient

**Model:** $\hat y = \text{softmax}(U^\top v_c)$, a probability for every word in the vocab of being the outside word.
**Loss:** cross-entropy with the one-hot truth $y$: $J = -\log \hat y_o$.

### The single most important gradient in deep learning
If $z$ are the scores (logits) and $J = \text{CrossEntropy}(\text{softmax}(z), y)$, then
$$\frac{\partial J}{\partial z} = \hat y - y$$
"prediction minus truth". This same result shows up in logistic regression, every classifier, and every language model's output layer.

Everything else is the chain rule through $z = U^\top v_c$:
* $\partial J/\partial v_c = U(\hat y - y)$ — each $u_w$ weighted by how wrong we were about word $w$.
* $\partial J/\partial u_w = (\hat y_w - y_w)\, v_c$, i.e. $(\hat y_o - 1)v_c$ for the true word, $\hat y_w v_c$ for the rest.

*Hint:* ~6 lines. Build `y_hat` with the provided `softmax`, make a one-hot `y`, then use the numpy column of the table in section 1.

In [ ]:
def naiveSoftmaxLossAndGradient(centerWordVec, outsideWordIdx, outsideVectors, dataset):
    """
    centerWordVec  : (d,)      v_c
    outsideWordIdx : int       o
    outsideVectors : (|V|, d)  rows are u_w   (= U transposed)
    dataset        : unused here (only needed for negative sampling)
    Returns loss (float), gradCenterVec (d,), gradOutsideVecs (|V|, d)
    """
    ### YOUR CODE HERE (~6 lines)
    y_hat = softmax(outsideVectors @ centerWordVec)        # (|V|,) predicted distribution
    loss = -np.log(y_hat[outsideWordIdx])                  # cross-entropy with one-hot y
    delta = y_hat.copy()
    delta[outsideWordIdx] -= 1                             # delta = y_hat - y   (no need to build y)
    gradCenterVec = delta @ outsideVectors                 # (|V|,) @ (|V|, d) -> (d,)
    gradOutsideVecs = np.outer(delta, centerWordVec)       # (|V|, d)
    ### END YOUR CODE
    return loss, gradCenterVec, gradOutsideVecs

### Small dummy setup for gradient checks
Same as the assignment's `getDummyObjects`: 5 words, 3-dimensional vectors.

In [ ]:
def getDummyObjects():
    dataset = type("dummy", (), {})()
    dataset.sampleTokenIdx = lambda: random.randint(0, 4)
    tokens = ["a", "b", "c", "d", "e"]
    dataset.getRandomContext = lambda C: (tokens[random.randint(0, 4)],
                                          [tokens[random.randint(0, 4)] for _ in range(2 * C)])
    random.seed(31415)
    np.random.seed(9265)
    dummy_vectors = normalizeRows(np.random.randn(10, 3))   # 5 center + 5 outside vectors
    dummy_tokens = {w: i for i, w in enumerate(tokens)}
    return dataset, dummy_vectors, dummy_tokens

def test_naive_softmax():
    ds, vecs, _ = getDummyObjects()
    outside = vecs[5:]
    gradcheck_naive(lambda v: naiveSoftmaxLossAndGradient(v, 1, outside, ds)[:2],
                    np.random.randn(3), "naiveSoftmax  dJ/dv_c")
    center = np.random.randn(3)
    gradcheck_naive(lambda U: naiveSoftmaxLossAndGradient(center, 1, U, ds)[::2],
                    outside.copy(), "naiveSoftmax  dJ/dU")
try_test(test_naive_softmax)

### Concept checks (the written questions, short versions)
Try answering yourself, then click to reveal.

<details><summary><b>(b-ii) When is ∂J/∂v_c = U(ŷ − y) equal to zero?</b></summary>

When $\hat y = y$ (the model puts probability 1 on the true word, which softmax can only approach, never reach), **or** more generally when $\hat y - y$ lies in the **null space** of $U$, i.e. the columns of $U$ weighted by $(\hat y - y)$ cancel out. Equivalently: $u_o = \sum_w \hat y_w u_w$, the true outside vector equals the model's *expected* outside vector.
</details>

<details><summary><b>(b-iii) Interpret the two terms when we do v_c ← v_c − α·gradient.</b></summary>

The update is $v_c \leftarrow v_c + \alpha\, u_o - \alpha \sum_w \hat y_w u_w$.
* $+u_o$ pulls $v_c$ **toward** the true outside word's vector, raising $u_o^\top v_c$.
* $-\sum_w \hat y_w u_w$ pushes $v_c$ **away** from what the model currently expects, especially from words it wrongly gives high probability.
</details>

<details><summary><b>(b-iv) When does L2 normalization lose useful info for sentiment?</b></summary>

Normalizing keeps the direction and throws away length. If $u_x = \alpha u_y$ with $\alpha > 0$, both words become the **same** vector after normalization. That's harmful if the magnitude carried meaning (e.g. "good" vs "phenomenal" pointing the same way with different strength, or a classifier that sums word vectors so length acts as a weight). If $\alpha < 0$ the directions are opposite, so normalization keeps them distinguishable (e.g. positive vs negative words) and nothing useful is lost.
</details>

## 4. Exercise 3 — negative sampling

Softmax needs a sum over the **whole vocabulary** (millions of words) for every training pair. Too slow.
Negative sampling replaces that with a handful of yes/no questions:
* "Is $o$ a real neighbor of $c$?" → want $\sigma(u_o^\top v_c)$ close to 1
* For $K$ random words $w_s$: "Is $w_s$ a real neighbor?" → want $\sigma(u_{w_s}^\top v_c)$ close to 0, i.e. $\sigma(-u_{w_s}^\top v_c)$ close to 1

$$J = -\log\sigma(u_o^\top v_c) - \sum_{s=1}^{K}\log\sigma(-u_{w_s}^\top v_c)$$

Using $\frac{d}{dx}[-\log\sigma(x)] = -(1-\sigma(x))$:
* $\partial J/\partial v_c = -(1-\sigma(u_o^\top v_c))\,u_o + \sum_s (1-\sigma(-u_{w_s}^\top v_c))\,u_{w_s}$
* $\partial J/\partial u_o = -(1-\sigma(u_o^\top v_c))\,v_c$
* $\partial J/\partial u_{w_s} = (1-\sigma(-u_{w_s}^\top v_c))\,v_c$ — **summed over every time $w_s$ was sampled** (part h)

### 3a. Readable loop version first
*Hint:* start from zeros, handle the true word, then loop over the negatives with `+=`. The `+=` is what makes duplicate samples count twice.

In [ ]:
def getNegativeSamples(outsideWordIdx, dataset, K):
    """Samples K indexes which are not the outsideWordIdx (given — do not modify)."""
    negSampleWordIndices = [None] * K
    for k in range(K):
        newidx = dataset.sampleTokenIdx()
        while newidx == outsideWordIdx:
            newidx = dataset.sampleTokenIdx()
        negSampleWordIndices[k] = newidx
    return negSampleWordIndices

def negSamplingLoop(centerWordVec, outsideWordIdx, outsideVectors, dataset, K=10):
    negSampleWordIndices = getNegativeSamples(outsideWordIdx, dataset, K)
    gradCenterVec = np.zeros_like(centerWordVec)
    gradOutsideVecs = np.zeros_like(outsideVectors)
    ### YOUR CODE HERE (~12 lines)
    # true outside word: want sigma(u_o . v_c) -> 1
    u_o = outsideVectors[outsideWordIdx]
    s_o = sigmoid(u_o @ centerWordVec)
    loss = -np.log(s_o)
    gradCenterVec += -(1 - s_o) * u_o
    gradOutsideVecs[outsideWordIdx] += -(1 - s_o) * centerWordVec
    # negatives: want sigma(-u_k . v_c) -> 1
    for k in negSampleWordIndices:
        u_k = outsideVectors[k]
        s_k = sigmoid(-u_k @ centerWordVec)
        loss += -np.log(s_k)
        gradCenterVec += (1 - s_k) * u_k
        gradOutsideVecs[k] += (1 - s_k) * centerWordVec   # += so duplicates add up
    ### END YOUR CODE
    return loss, gradCenterVec, gradOutsideVecs

In [ ]:
def test_neg_loop():
    ds, vecs, _ = getDummyObjects()
    outside = vecs[5:]
    gradcheck_naive(lambda v: negSamplingLoop(v, 1, outside, ds)[:2],
                    np.random.randn(3), "negSamplingLoop  dJ/dv_c")
    center = np.random.randn(3)
    gradcheck_naive(lambda U: negSamplingLoop(center, 1, U, ds)[::2],
                    outside.copy(), "negSamplingLoop  dJ/dU")
try_test(test_neg_loop)

### 3b. Vectorized version (what the autograder wants — it checks efficiency)

The trick from part (g-ii): stack the vectors and give each row a sign, $+1$ for the true word, $-1$ for negatives. Then every row has the same form, $-\log\sigma(\text{sign}_k \cdot u_k^\top v_c)$, and its derivative w.r.t. the score $z_k = u_k^\top v_c$ is

$$\frac{\partial J}{\partial z_k} = -\,\text{sign}_k\,(1-\sigma(\text{sign}_k z_k))$$

Compute that vector once (the reused quantity) and both gradients fall out of it.

**The library call you need: `np.add.at`.** `grad[indices] += rows` looks right but is **wrong** when `indices` has duplicates: numpy writes each duplicate once, so a word sampled 3 times only gets counted once. `np.add.at(grad, indices, rows)` accumulates correctly. (PyTorch's equivalent is `index_add_`, and autograd does it for you automatically.)

*Hint:* ~8 lines. `U = outsideVectors[indices]` has shape `(K+1, d)`.

In [ ]:
def negSamplingLossAndGradient(centerWordVec, outsideWordIdx, outsideVectors, dataset, K=10):
    negSampleWordIndices = getNegativeSamples(outsideWordIdx, dataset, K)
    indices = [outsideWordIdx] + negSampleWordIndices
    ### YOUR CODE HERE (~8 lines, no Python loops)
    U = outsideVectors[indices]                     # (K+1, d)  row 0 = u_o, rest = negatives
    signs = np.array([1.0] + [-1.0] * K)            # (K+1,)
    s = sigmoid(signs * (U @ centerWordVec))        # (K+1,)  sigma(+u_o.v_c), sigma(-u_k.v_c)
    loss = -np.sum(np.log(s))
    dz = -signs * (1 - s)                           # dJ/d(u_k . v_c)  <- the reusable quantity
    gradCenterVec = dz @ U                          # (K+1,) @ (K+1, d) -> (d,)
    gradOutsideVecs = np.zeros_like(outsideVectors)
    np.add.at(gradOutsideVecs, indices, np.outer(dz, centerWordVec))   # duplicates accumulate
    ### END YOUR CODE
    return loss, gradCenterVec, gradOutsideVecs

In [ ]:
def test_neg_vectorized():
    ds, vecs, _ = getDummyObjects()
    outside = vecs[5:]
    gradcheck_naive(lambda v: negSamplingLossAndGradient(v, 1, outside, ds)[:2],
                    np.random.randn(3), "negSampling  dJ/dv_c")
    center = np.random.randn(3)
    gradcheck_naive(lambda U: negSamplingLossAndGradient(center, 1, U, ds)[::2],
                    outside.copy(), "negSampling  dJ/dU")
    # must agree exactly with the loop version
    state = random.getstate(); a = negSamplingLoop(center, 1, outside, ds)
    random.setstate(state);    b = negSamplingLossAndGradient(center, 1, outside, ds)
    assert all(np.allclose(x, y) for x, y in zip(a, b)), "vectorized != loop version"
    print("✅ vectorized matches the loop version")
try_test(test_neg_vectorized)

**See the duplicate bug with your own eyes:**

In [ ]:
g = np.zeros(4)
idx = [2, 2, 2]
g[idx] += 1;            print("fancy-index +=  :", g)   # word 2 counted ONCE  (bug)
g = np.zeros(4)
np.add.at(g, idx, 1);   print("np.add.at       :", g)   # word 2 counted 3 times (correct)

## 5. Exercise 4 — skip-gram over a whole window

The window loss is just the sum of pair losses (part i of the written section):
$$J_{\text{skip-gram}} = \sum_{-m \le j \le m,\ j\ne 0} J(v_c, w_{t+j}, U)$$
So the gradients are sums too. Only the center word's row of $V$ gets a gradient; every other row of $V$ gets 0.

*Hint:* ~6 lines. Look up the center index, loop over `outsideWords`, accumulate loss, add to `gradCenterVecs[centerIdx]`, add the full outside gradient.

In [ ]:
def skipgram(currentCenterWord, windowSize, outsideWords, word2Ind,
             centerWordVectors, outsideVectors, dataset,
             word2vecLossAndGradient=naiveSoftmaxLossAndGradient):
    loss = 0.0
    gradCenterVecs = np.zeros(centerWordVectors.shape)
    gradOutsideVectors = np.zeros(outsideVectors.shape)
    ### YOUR CODE HERE (~6 lines)
    c = word2Ind[currentCenterWord]
    v_c = centerWordVectors[c]
    for w in outsideWords:
        l, g_c, g_out = word2vecLossAndGradient(v_c, word2Ind[w], outsideVectors, dataset)
        loss += l
        gradCenterVecs[c] += g_c
        gradOutsideVectors += g_out
    ### END YOUR CODE
    return loss, gradCenterVecs, gradOutsideVectors

In [ ]:
def word2vec_sgd_wrapper(word2vecModel, word2Ind, wordVectors, dataset, windowSize,
                         word2vecLossAndGradient=naiveSoftmaxLossAndGradient, batchsize=50):
    """Given (same as the assignment). wordVectors stacks [center vectors; outside vectors]."""
    loss = 0.0
    grad = np.zeros(wordVectors.shape)
    N = wordVectors.shape[0]
    centerWordVectors = wordVectors[:N // 2, :]
    outsideVectors = wordVectors[N // 2:, :]
    for _ in range(batchsize):
        windowSize1 = random.randint(1, windowSize)
        centerWord, context = dataset.getRandomContext(windowSize1)
        c, gin, gout = word2vecModel(centerWord, windowSize1, context, word2Ind,
                                     centerWordVectors, outsideVectors, dataset,
                                     word2vecLossAndGradient)
        loss += c / batchsize
        grad[:N // 2, :] += gin / batchsize
        grad[N // 2:, :] += gout / batchsize
    return loss, grad

def test_skipgram():
    ds, vecs, toks = getDummyObjects()
    gradcheck_naive(lambda v: word2vec_sgd_wrapper(skipgram, toks, v, ds, 5, naiveSoftmaxLossAndGradient),
                    vecs.copy(), "skip-gram + naive softmax")
    gradcheck_naive(lambda v: word2vec_sgd_wrapper(skipgram, toks, v, ds, 5, negSamplingLossAndGradient),
                    vecs.copy(), "skip-gram + negative sampling")
try_test(test_skipgram)

## 6. Exercise 5 — SGD

$$x \leftarrow x - \alpha \nabla_x J$$
That's the whole optimizer. We also halve the learning rate every `anneal_every` steps, like `sgd.py`.

*Hint:* 2 lines: call `f(x)` to get `(loss, grad)`, then update `x`.

In [ ]:
def sgd(f, x0, step, iterations, print_every=200, anneal_every=1000):
    x = x0.copy()
    exploss, history = None, []
    for it in range(1, iterations + 1):
        ### YOUR CODE HERE (~2 lines)
        loss, grad = f(x)
        x -= step * grad
        ### END YOUR CODE
        exploss = loss if exploss is None else 0.95 * exploss + 0.05 * loss   # smoothed loss
        if it % print_every == 0:
            history.append(exploss)
            print(f"iter {it:5d}   loss {exploss:.4f}")
        if it % anneal_every == 0:
            step *= 0.5
    return x, history

In [ ]:
def test_sgd():
    quad = lambda x: (np.sum(x ** 2), 2 * x)     # minimum at 0
    for x0 in [0.0, 1.5, -4.0]:
        x, _ = sgd(quad, np.array(x0), 0.01, 1000, print_every=10**9)
        assert abs(x) < 1e-3, f"didn't converge from {x0}: got {x}"
    print("✅ sgd converges on a quadratic")
try_test(test_sgd)

## 7. Show time — train on the toy corpus (≈1 minute)
Same recipe as `run.py`: small random center vectors, zero outside vectors, negative sampling, window 3.

In [ ]:
random.seed(314); np.random.seed(42)
V, d = len(dataset.words), 10
wordVectors = np.concatenate([(np.random.rand(V, d) - 0.5) / d, np.zeros((V, d))], axis=0)

wordVectors, history = sgd(
    lambda vec: word2vec_sgd_wrapper(skipgram, dataset.tokens, vec, dataset, 3,
                                     negSamplingLossAndGradient),
    wordVectors, step=0.3, iterations=3000, print_every=300, anneal_every=1500)

plt.plot(np.arange(1, len(history) + 1) * 300, history); plt.xlabel("iteration"); plt.ylabel("smoothed loss")
plt.title("Training loss"); plt.show()

### Look at what it learned
We use center + outside vectors added together (a common trick), project to 2D with PCA (an SVD), and list nearest neighbors by **cosine similarity** (dot product of L2-normalized vectors — the standard way to compare embeddings).

In [ ]:
emb = wordVectors[:V] + wordVectors[V:]

def most_similar(word, vecs, ds, topn=5):
    W = normalizeRows(vecs)
    sims = W @ W[ds.tokens[word]]
    order = np.argsort(-sims)
    return [(ds.words[i], round(float(sims[i]), 3)) for i in order if ds.words[i] != word][:topn]

for w in ["great", "terrible", "king", "coffee", "woman"]:
    print(f"{w:9s} -> {most_similar(w, emb, dataset)}")

show = ["great", "amazing", "wonderful", "excellent", "brilliant",
        "boring", "awful", "terrible", "dull", "bad",
        "king", "queen", "prince", "princess", "man", "woman", "boy", "girl",
        "tea", "coffee", "juice", "milk", "movie", "film", "story", "show"]
X = emb[[dataset.tokens[w] for w in show]]
X = X - X.mean(axis=0)
_, _, Vt = np.linalg.svd(X, full_matrices=False)
P = X @ Vt[:2].T
plt.figure(figsize=(8, 6))
for (x, y), w in zip(P, show):
    plt.scatter(x, y, s=10); plt.annotate(w, (x, y), fontsize=10)
plt.title("word vectors, PCA to 2D"); plt.show()

# zoom in on the sentiment words (redo PCA on just them so they spread out)
sent = show[:10]
Xs = emb[[dataset.tokens[w] for w in sent]]; Xs = Xs - Xs.mean(axis=0)
Ps = Xs @ np.linalg.svd(Xs, full_matrices=False)[2][:2].T
plt.figure(figsize=(6, 4))
for i, ((x, y), w) in enumerate(zip(Ps, sent)):
    plt.scatter(x, y, c="tab:green" if i < 5 else "tab:red"); plt.annotate(w, (x, y))
plt.title("zoom: positive (green) vs negative (red)"); plt.show()

cos = lambda a, b: float(normalizeRows(emb)[dataset.tokens[a]] @ normalizeRows(emb)[dataset.tokens[b]])
print(f"cos(great, terrible) = {cos('great', 'terrible'):.2f}   <- antonyms, but similar usage")
print(f"cos(great, coffee)   = {cos('great', 'coffee'):.2f}")

**What to notice (this is the kind of thing part 2c asks you to write):** groups that share contexts cluster (drinks, royals, people, film nouns). Positive and negative adjectives sit fairly *close* to each other because they appear in the same slots ("the movie was ___"). word2vec learns **"used the same way"**, which is not the same as **"means the same thing"**. That's exactly why antonyms like "boring" and "great" end up near each other in the real SST plot too.

---
## 8. The industry way — same model, with libraries

Nobody hand-writes gradients at work. You write the **forward pass** (the loss), and autograd computes the gradients. Your hand-derived formulas are still valuable: they tell you *what* autograd is computing, which is how you debug weird training.

### 8a. Verify your gradient with PyTorch autograd
`F.cross_entropy` = softmax + negative log likelihood, in one numerically stable call.

In [ ]:
try:
    import torch
    import torch.nn.functional as F

    torch.manual_seed(0)
    Vs, ds_ = 7, 4
    vc = torch.randn(ds_, dtype=torch.float64, requires_grad=True)
    U  = torch.randn(Vs, ds_, dtype=torch.float64, requires_grad=True)   # rows = u_w, like our code
    o = 3

    loss = F.cross_entropy((U @ vc).unsqueeze(0), torch.tensor([o]))    # the forward pass is ALL you write
    loss.backward()                                                      # autograd does the calculus

    my_loss, my_gc, my_gU = naiveSoftmaxLossAndGradient(vc.detach().numpy(), o, U.detach().numpy(), None)
    print("loss matches       :", np.isclose(my_loss, loss.item()))
    print("dJ/dv_c matches    :", np.allclose(my_gc, vc.grad.numpy()))
    print("dJ/dU   matches    :", np.allclose(my_gU, U.grad.numpy()))
except ImportError:
    print("PyTorch not installed. It's preinstalled in Colab; locally run:  pip install torch")

### 8b. Skip-gram with negative sampling as a PyTorch module
This is what production-style code looks like: `nn.Embedding` is just a lookup table (our `centerWordVectors` / `outsideVectors`), `F.logsigmoid` is a stable $\log\sigma$, and training is batched with Adam.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class SkipGramNS(nn.Module):
        def __init__(self, vocab_size, dim):
            super().__init__()
            self.center = nn.Embedding(vocab_size, dim)    # V
            self.outside = nn.Embedding(vocab_size, dim)   # U^T
            nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
            nn.init.zeros_(self.outside.weight)

        def forward(self, c, o, negs):                     # c, o: (B,)   negs: (B, K)
            v_c = self.center(c)                           # (B, d)
            pos = (self.outside(o) * v_c).sum(-1)          # (B,)    u_o . v_c
            neg = torch.einsum("bkd,bd->bk", self.outside(negs), v_c)   # (B, K)  u_k . v_c
            return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(-1)).mean()

    # build every (center, outside) pair with window 3
    pairs = [(dataset.tokens[s[i]], dataset.tokens[s[j]])
             for s in dataset.sentences for i in range(len(s))
             for j in range(max(0, i - 3), min(len(s), i + 4)) if j != i]
    pairs = torch.tensor(pairs)
    counts = torch.bincount(pairs[:, 0], minlength=V).double()
    noise = counts ** 0.75 / (counts ** 0.75).sum()        # unigram^(3/4) for negatives

    torch.manual_seed(0)
    model = SkipGramNS(V, 10)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    K, B = 10, 512
    for epoch in range(5):
        perm = torch.randperm(len(pairs))
        for i in range(0, len(pairs), B):
            batch = pairs[perm[i:i + B]]
            negs = torch.multinomial(noise, len(batch) * K, replacement=True).view(len(batch), K)
            loss = model(batch[:, 0], batch[:, 1], negs)
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"epoch {epoch}  loss {loss.item():.4f}")

    torch_emb = (model.center.weight + model.outside.weight).detach().numpy()
    for w in ["great", "king", "coffee"]:
        print(f"{w:7s} -> {most_similar(w, torch_emb, dataset)}")
except ImportError:
    print("PyTorch not installed. It's preinstalled in Colab; locally run:  pip install torch")

### 8c. The one-liner: gensim
If you just need word2vec vectors for a project, this is it. `sg=1` means skip-gram, `negative=10` means negative sampling with K=10.

In [ ]:
try:
    from gensim.models import Word2Vec
    w2v = Word2Vec(sentences=dataset.sentences, vector_size=10, window=3, sg=1,
                   negative=10, min_count=1, epochs=20, seed=0, workers=1)
    for w in ["great", "king", "coffee"]:
        print(f"{w:7s} ->", [(x, round(s, 3)) for x, s in w2v.wv.most_similar(w, topn=5)])
except ImportError:
    print("gensim not installed. Run:  pip install gensim   (then restart the kernel)")

---
## 9. Cheat sheet — what to actually remember

| Remember this | Why it matters at work |
|---|---|
| softmax + cross-entropy gradient $= \hat y - y$ | Output layer of every classifier and LLM |
| $\sigma'(x) = \sigma(x)(1-\sigma(x))$, $\ \frac{d}{dx}[-\log\sigma(x)] = -(1-\sigma(x))$ | Binary classification, logistic regression |
| Gradient has the same shape as the parameter | #1 debugging rule |
| Negative sampling = push real pairs together, random pairs apart | Same idea as contrastive learning: retrieval / RAG embeddings, CLIP, recommender two-tower models |
| Softmax over a huge vocab is expensive → sample instead | Why sampled losses and in-batch negatives exist |
| Cosine similarity = dot product of L2-normalized vectors | How vector databases and semantic search compare embeddings |
| Normalizing throws away vector length | Sometimes length carries signal (frequency, confidence) |
| Duplicate indices: `np.add.at` / `index_add_`, never `x[idx] += ...` | A silent bug that shows up in real code |
| Gradient checking with finite differences | How to test any custom loss or layer |
| Embeddings capture *usage*, not *meaning* | Antonyms end up close; biases in data end up in vectors |

You do **not** need to memorize the full derivations. You *should* be able to re-derive $\hat y - y$ on a whiteboard in two minutes, because interviewers do ask.